In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn import svm
from sklearn.pipeline import make_pipeline, Pipeline, clone
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.metrics import f1_score, accuracy_score

df_mine = pd.read_csv('XY_Stellar.csv')

meta_col = df_mine.select_dtypes("int64").columns

df_train = df_mine[df_mine.isnull()['Class']==False].drop(columns=meta_col)

df_test = df_mine[df_mine.isnull()['Class']==True].drop(columns="Class").drop(columns=meta_col)

X = df_train.drop(columns='Class')
Y = df_train['Class']

clean = True

if clean:
    #clean up outliers and unrealistic data for each class
    cols_to_clean = ['redshift', 'u', 'g', 'r', 'i', 'z']

    # Group by the target labels (Y) and compute quantiles broadcasted to row shapes
    grouped = X[cols_to_clean].groupby(Y)
    lower = grouped.transform(lambda g: g.quantile(0.0001))
    upper = grouped.transform(lambda g: g.quantile(0.9999))

    # Build the boolean mask across all relevant columns
    mask1 = (X[cols_to_clean] >= lower) & (X[cols_to_clean] <= upper)

    masku = (X['u'] >= 10) & (X['u'] <= 28)
    maskg = (X['g'] >= 10) & (X['g'] <= 28)
    maskr = (X['r'] >= 9.0) & (X['u'] <= 27)
    maski = (X['i'] >= 9.0) & (X['u'] <= 27)
    maskz = (X['z'] >= 9.0) & (X['u'] <= 27)
    mask_redshift = (X['redshift'] >= -0.01) & (X['redshift'] <= 8.0)

    valid_rows = mask1.all(axis=1) & masku & maskg & maskr & maski & maskz & mask_redshift

    # Filter both feature and target sets
    X_clean = X[valid_rows]
    Y_clean = Y[valid_rows]

  #get a shuffled split for plotting
X_train, X_test, Y_train, Y_test = train_test_split(X_clean, Y_clean, test_size=0.1, random_state=42, stratify=Y_clean)

In [2]:
# @title
class Segmentedclassifer(BaseEstimator, ClassifierMixin):
    def __init__(self, preprocessor, estimator, segment_target=['redshift'], n_bins=24, min_samples_per_bin=500):
        self.preprocessor = preprocessor
        self.estimator = estimator
        self.segment_target = segment_target
        self.n_bins = n_bins
        self.min_samples_per_bin = min_samples_per_bin
        self.global_fit = False

    def _get_group_keys(self, X_df, is_fit=False):
        keys = []
        for col in self.segment_target:
            if pd.api.types.is_numeric_dtype(X_df[col]):
                if is_fit:
                    # Calculate and store interior bin edges
                    vals = X_df[col].to_numpy()
                    edges = np.unique(np.quantile(vals, np.linspace(0, 1, self.n_bins + 1)))[1:-1]
                    self.bin_edges_[col] = edges

                # Digitize into integer bins (0 to len(edges))
                edges = self.bin_edges_[col]
                binned = pd.Series(np.digitize(X_df[col].to_numpy(), edges), index=X_df.index)
                keys.append(binned)
            else:
                # Categorical / string columns group by their raw values
                keys.append(X_df[col])
        return keys

    def fit(self, X, y):
        X_df = X.copy()
        # Preserve original indices for clean alignment
        y_arr = np.asarray(y).ravel()
        y_trans = pd.Series(y_arr,index=X_df.index)
        self.bin_edges_ = {}
        group_keys = self._get_group_keys(X_df, is_fit=True)
        self.models_ = {}
        for comb_tuple, group in X_df.groupby(group_keys, observed=True):
            n_samples = len(group)
            if n_samples >= self.min_samples_per_bin:
                y_seg = y_trans[group.index]
                pipe = make_pipeline(clone(self.preprocessor), clone(self.estimator))
                pipe.fit(group, y_seg)

                self.models_[comb_tuple] = pipe
            else:
                if self.global_fit:
                    self.models_[comb_tuple] = self.global_pipe_
                else:
                    # Global fallback model and bounds
                    X_small, _, Y_small, __ = train_test_split(X_df, y_trans, test_size=0.8, random_state=42, stratify=y_trans)
                    self.global_pipe_ = make_pipeline(clone(self.preprocessor), clone(self.estimator))
                    self.global_pipe_.fit(X_small, Y_small)
                    self.global_fit = True
                    self.models_[comb_tuple] = self.global_pipe_


        return self

    def predict(self, X):
        print('predicting')
        # Reset index locally to guarantee 0..N-1 positional indexing
        X_df = X.reset_index(drop=True)
        preds = np.empty(len(X_df), dtype=str)

        group_keys = self._get_group_keys(X_df, is_fit=False)

        for comb_tuple, group in X_df.groupby(group_keys, observed=True):
            if len(group) == 0:
                continue

            # model = self.models_.get(comb_tuple, self.global_pipe_)
            model = self.models_.get(comb_tuple)

            preds[group.index] = model.predict(group)

        return preds

In [3]:
# @title
class FeatureCreator(BaseEstimator, TransformerMixin):
    GENERATED_FEATURES = [
        "r_i",
        "g_r",
        "i_z",
        "u_r",
        "u_g",
        # "g_z"

    ]

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        else:
            self.n_features_in_ = X.shape[1]

        return self

    def transform(self, X):
        df = X.copy()
        df["r_i"] = df["r"]-df["i"]
        df["g_r"] = df["g"]-df["r"]
        df["i_z"] = df["i"]-df["z"]
        df["u_r"] = df["u"]-df["r"]
        df["u_g"] = df["u"]-df["g"]
        # df["g_z"] = df["g"]-df["z"]

        return df
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            if not hasattr(self, "feature_names_in_"):
                raise ValueError(
                    "input_features must be provided when fitting on "
                    "data without column names."
                )
            input_features = self.feature_names_in_

        input_features = np.asarray(input_features, dtype=object)

        return np.concatenate([
            input_features,
            np.asarray(self.GENERATED_FEATURES, dtype=object),
        ])

In [4]:
class ColorRedshiftInteractionAdder(BaseEstimator, TransformerMixin):
    """
    Adds color-redshift interaction (CRI): redshift * (sum of color
    differences). Assumes color_cols already exist in X (i.e. add
    color indices earlier in the pipeline). Stateless.
    """
    def __init__(self, color_cols, redshift_col='redshift'):
        self.color_cols = color_cols
        self.redshift_col = redshift_col

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        missing = [c for c in self.color_cols if c not in X.columns]
        if missing:
            raise ValueError(f"color_cols not found in X: {missing} - add color indices earlier in the pipeline")
        if self.redshift_col not in X.columns:
            raise ValueError(f"'{self.redshift_col}' not found - cri needs it present")
        return self

    def transform(self, X):
        df = X.copy()
        df['cri'] = df[self.redshift_col] * df[self.color_cols].sum(axis=1)
        return df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)
        return np.concatenate([input_features, np.asarray(['cri'], dtype=object)])

In [5]:
class NormalizedMagnitudeAdder(BaseEstimator, TransformerMixin):
    """
    Adds normalized-magnitude ratios: band/ref for each band in `bands`.
    Stateless - safe to apply identically to train and eval.
    """
    def __init__(self, ref='r', bands=('u', 'i', 'z'), drop_original=False):
        self.ref = ref
        self.bands = bands
        self.drop_original = drop_original

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        df = X.copy()
        for b in self.bands:
            df[f'{b}/{self.ref}'] = df[b] / df[self.ref]

        if self.drop_original:
            cols_to_drop = [b for b in list(self.bands) + [self.ref] if b in df.columns]
            df = df.drop(columns=cols_to_drop)

        return df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)

        new_cols = np.asarray([f'{b}/{self.ref}' for b in self.bands], dtype=object)
        out = np.concatenate([input_features, new_cols])

        if self.drop_original:
            drop = set(list(self.bands) + [self.ref])
            out = np.asarray([c for c in out if c not in drop], dtype=object)

        return out

In [6]:
class NormalizedFluxRatioAdder(BaseEstimator, TransformerMixin):
    """
    Adds normalized flux ratio (NFR): num / (denom + epsilon).
    Stateless - safe to apply identically to train and eval.
    """
    def __init__(self, num='u', denom='z', epsilon=1e-3, drop_original=False):
        self.num = num
        self.denom = denom
        self.epsilon = epsilon
        self.drop_original = drop_original

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        for c in (self.num, self.denom):
            if c not in X.columns:
                raise ValueError(f"'{c}' not found in X")
        return self

    def transform(self, X):
        df = X.copy()
        df['nfr'] = df[self.num] / (df[self.denom] + self.epsilon)

        if self.drop_original:
            cols_to_drop = [c for c in (self.num, self.denom) if c in df.columns]
            df = df.drop(columns=cols_to_drop)

        return df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)

        out = np.concatenate([input_features, np.asarray(['nfr'], dtype=object)])
        if self.drop_original:
            drop = {self.num, self.denom}
            out = np.asarray([c for c in out if c not in drop], dtype=object)
        return out

In [7]:
class CRINFRComposer(BaseEstimator, TransformerMixin):
    """
    Adds the CRI*NFR composite feature (or its log1p), combining
    color-redshift interaction and normalized flux ratio. Requires
    cri_col and nfr_col to already exist in X. Stateless.
    """
    def __init__(self, cri_col='cri', nfr_col='nfr', log1p=False, drop_original=False):
        self.cri_col = cri_col
        self.nfr_col = nfr_col
        self.log1p = log1p
        self.drop_original = drop_original

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        for c in (self.cri_col, self.nfr_col):
            if c not in X.columns:
                raise ValueError(f"'{c}' not found in X - add CRI/NFR earlier in the pipeline")
        return self

    def transform(self, X):
        df = X.copy()
        product = df[self.cri_col] * df[self.nfr_col]
        self.out_col_ = 'log1p_cri_nfr' if self.log1p else 'cri_nfr'

        if self.log1p:
            product = product.clip(lower=-1 + 1e-6)
            df[self.out_col_] = np.log1p(product)
        else:
            df[self.out_col_] = product

        if self.drop_original:
            cols_to_drop = [c for c in (self.cri_col, self.nfr_col) if c in df.columns]
            df = df.drop(columns=cols_to_drop)

        return df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)

        out_col = 'log1p_cri_nfr' if self.log1p else 'cri_nfr'
        out = np.concatenate([input_features, np.asarray([out_col], dtype=object)])
        if self.drop_original:
            drop = {self.cri_col, self.nfr_col}
            out = np.asarray([c for c in out if c not in drop], dtype=object)
        return out

In [8]:
class ObjectSizeAdder(BaseEstimator, TransformerMixin):
    """
    Adds an object-size feature: angular spread of RA/Dec from the
    origin, scaled by redshift. Stateless - safe to apply identically
    to train and eval.
    """
    def __init__(self, log=True, epsilon=1e-6):
        self.log = log
        self.epsilon = epsilon

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        for c in ('alpha', 'delta', 'redshift'):
            if c not in X.columns:
                raise ValueError(f"'{c}' not found in X")
        return self

    def transform(self, X):
        df = X.copy()
        angular_spread = np.sqrt(df['alpha']**2 + df['delta']**2)
        size = angular_spread * df['redshift']

        self.out_col_ = 'log_size' if self.log else 'size'
        if self.log:
            df[self.out_col_] = np.log(size.clip(lower=self.epsilon))
        else:
            df[self.out_col_] = size

        return df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)
        out_col = 'log_size' if self.log else 'size'
        return np.concatenate([input_features, np.asarray([out_col], dtype=object)])

In [9]:
class ColumnPolynomialFeatures(BaseEstimator, TransformerMixin):
    """
    Applies PolynomialFeatures to a specified subset of columns only,
    leaving all other columns untouched. Replaces the selected raw
    columns with their polynomial expansion (which already includes the
    degree-1 terms), so there's no duplication between raw and expanded
    versions of the same column.
    """
    def __init__(self, columns, degree=2, interaction_only=False, include_bias=False):
        self.columns = columns
        self.degree = degree
        self.interaction_only = interaction_only
        self.include_bias = include_bias

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        missing = [c for c in self.columns if c not in X.columns]
        if missing:
            raise ValueError(f"columns not found in X: {missing}")

        self.other_cols_ = [c for c in X.columns if c not in self.columns]
        self.poly_ = PolynomialFeatures(
            degree=self.degree,
            interaction_only=self.interaction_only,
            include_bias=self.include_bias,
        )
        self.poly_.fit(X[self.columns])
        self.poly_feature_names_ = self.poly_.get_feature_names_out(self.columns)
        return self

    def transform(self, X):
        df = X.copy()
        poly_arr = self.poly_.transform(df[self.columns])
        poly_df = pd.DataFrame(poly_arr, columns=self.poly_feature_names_, index=df.index)

        # keep untouched columns as-is, replace selected columns with their poly expansion
        out = pd.concat([df[self.other_cols_], poly_df], axis=1)
        return out

    def get_feature_names_out(self, input_features=None):
        return np.concatenate([
            np.asarray(self.other_cols_, dtype=object),
            np.asarray(self.poly_feature_names_, dtype=object),
        ])

In [10]:
full_preprocessor = Pipeline([
                                ("feature_add", FeatureCreator()),
                                ("object_size", ObjectSizeAdder()),
                                ("cri", ColorRedshiftInteractionAdder(
                                    color_cols=["r_i","g_r","i_z","u_r","u_g"],
                                    redshift_col='redshift'
                                )),
                                ("nfr", NormalizedFluxRatioAdder(num='u', denom='z', epsilon=1e-3)),
                                ("cri_nfr", CRINFRComposer(cri_col='cri', nfr_col='nfr', log1p=True)),
                                ("scaler", StandardScaler())
                            ])

full_preprocessor.set_output(transform="pandas")

model = Segmentedclassifer(
    preprocessor=full_preprocessor,
    estimator=svm.SVC(kernel='rbf',C=22.57,gamma=0.15),
    segment_target=['redshift'],
    #9=0.9729
    #8=0.97249
    #7=0.9730
    #5=0.972
    #3=0.970
    #2=0.970
    #1=0.969 no bin, 1 model
    n_bins=7,
    min_samples_per_bin=500
)

model.fit(X_train, Y_train)
y_val_pred = model.predict(X_test)
print("accu:", accuracy_score(Y_test,y_val_pred))
print("f1:", f1_score(Y_test,y_val_pred,average='macro'))


predicting
accu: 0.9730561970746728
f1: 0.9703964119573675


1. **Title**: Segmented SVC model on redshift
1. **Idea Discovery**: Pairwise feature scatter plots across target classes were conducted during exploratory data analysis (EDA). The class distributions were distinct along the redshift feature, with targets composition varying substantially across different redshift intervals. While galaxies and quasars exhibit significant overlap across several photometric features, they display distinctly different cluster shapes and heavy tails. Domain-informed data inspection also revealed physically implausible measurement values (Abdurro’uf et al., 2022).
1. **Finding's Importance**: $redshift$ was primarily the most predictive feature, capable of isolating over 90% of class objects on its own. However, discriminating between galaxies and quasars was significantly more difficult. Furthermore, many feature pairs display non-linear, irregularly distributed boundaries, which means a classifier that work with non-linear decision boundaries, such as Support Vector Classification (SVC), would be a strong candidate (Géron, 2022, Chapter 5). A domain-aware data trimming strategy was also required to filter out valid numeric values that represent impossible physical values.
1. **Experiment Setup**: The dataset was first trimmed, against several numeric features, including ($redshift, u, g, r, i, z$), regarding the possible value ranges. Logistic Regression, Linear Discriminant Analysis (LDA), Quadratic Discriminant Analysis (QDA), and SVC were then evaluated on a representative subset via 10-fold cross-validation. The top-performing architecture underwent randomized hyperparameter tuning. Finally, the optimal model and parameters served as the base estimator within a custom segmented classifier, evaluated across varying redshift bin configurations.
1. **Results**:
- Baseline Comparison: The baseline SVC achieved an initial accuracy of 0.95 without segmentation, outperforming LDA, QDA, and Logistic Regression (0.93–0.94).

- Hyperparameter Tuning: Tuning an RBF kernel (optimizing $C$ and $\gamma$) increased performance to 0.96. However, standard SVC training exhibited its known drawback of super-linear training time scaling as sample size grew.

- Segmented SVC: Partitioning the dataset across redshift intervals improved accuracy to 0.97 while significantly reducing computational runtime, as each regional sub-estimator trained on a substantially smaller data slice.